# Machine Learning Final Project
Scholarship Prediction System Development

This notebook implements the final project pipeline manually. It avoids scikit-learn training APIs.

## 1. Setup and Imports

In [ ]:
from __future__ import annotations
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid')

MIDTERM_BASE_FEATURES = ['gpa','attendance_rate','study_hours_per_week','exam_score','household_income']
FINAL_BASE_FEATURES = MIDTERM_BASE_FEATURES + ['part_time_job_hours']
ENGINEERED_FEATURES = ['academic_strength','study_efficiency','income_per_study_hour','work_study_balance']
IMPROVED_FEATURES = FINAL_BASE_FEATURES + ENGINEERED_FEATURES

print('Current working folder:', Path.cwd().resolve())

## 2. File Paths

In [ ]:
# Keep the notebook and CSV files in the same folder.
# This is the same simple style used in the midterm notebook.

PROJECT_DIR = Path.cwd()
FINAL_DIR = PROJECT_DIR

TRAIN_PATH = 'ml_dataset_v2_train.csv'
DEV_PATH = 'ml_dataset_v2_dev.csv'
HIDDEN_PATH = 'hidden_test_data.csv'

print('Project folder:', PROJECT_DIR.resolve())
print('Train CSV:', TRAIN_PATH)
print('Dev CSV:', DEV_PATH)
print('Hidden/Test CSV:', HIDDEN_PATH)

## 3. Load and Inspect Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)
hidden_df = pd.read_csv(HIDDEN_PATH)

print('Train shape:', train_df.shape)
print('Dev shape:', dev_df.shape)
print('Hidden test shape:', hidden_df.shape)
display(train_df.head())

## 4. Exploratory Data Analysis

In [ ]:
print('Train label distribution:')
print(train_df['label'].value_counts().sort_index())
print('\nMissing values in train:', int(train_df.isna().sum().sum()))
print('Missing values in dev:', int(dev_df.isna().sum().sum()))
display(train_df[FINAL_BASE_FEATURES + ['label']].describe().T)

plt.figure(figsize=(6,4))
sns.countplot(data=train_df, x='label', hue='label', legend=False)
plt.title('Target Distribution')
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(train_df[FINAL_BASE_FEATURES + ['label']].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlation Heatmap')
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(data=train_df, x='label', y='gpa', hue='label', legend=False)
plt.title('GPA by Scholarship Label')
plt.show()

## 5. Data Validation and Feature Engineering Helpers

In [ ]:
def validate_columns(df: pd.DataFrame, required_cols: list[str], name: str) -> None:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def feature_medians_from_train(train_df: pd.DataFrame) -> dict[str, float]:
    medians: dict[str, float] = {}
    for col in FINAL_BASE_FEATURES:
        if col in train_df.columns:
            medians[col] = float(train_df[col].median())
        else:
            medians[col] = 0.0
    return medians


def build_feature_frame(
    df: pd.DataFrame,
    medians: dict[str, float],
    include_engineered: bool,
    feature_cols: list[str] | None = None,
) -> pd.DataFrame:
    work = df.copy()
    for col in FINAL_BASE_FEATURES:
        if col not in work.columns:
            work[col] = medians[col]
        work[col] = pd.to_numeric(work[col], errors="coerce").fillna(medians[col])

    if include_engineered:
        work["academic_strength"] = work["gpa"] * work["exam_score"]
        work["study_efficiency"] = work["exam_score"] / (work["study_hours_per_week"] + 1.0)
        work["income_per_study_hour"] = work["household_income"] / (
            work["study_hours_per_week"] + 1.0
        )
        work["work_study_balance"] = work["study_hours_per_week"] - work["part_time_job_hours"]

    selected = feature_cols if feature_cols is not None else (
        IMPROVED_FEATURES if include_engineered else FINAL_BASE_FEATURES
    )
    return work[selected].astype(float)


## 6. Scaler

In [ ]:
class StandardScalerModel:
    def __init__(self, eps: float = 1e-8):
        self.eps = eps
        self.mean_: np.ndarray | None = None
        self.std_: np.ndarray | None = None

    def fit(self, x: np.ndarray) -> "StandardScalerModel":
        self.mean_ = x.mean(axis=0)
        self.std_ = x.std(axis=0)
        return self

    def transform(self, x: np.ndarray) -> np.ndarray:
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError("Scaler has not been fitted.")
        return (x - self.mean_) / (self.std_ + self.eps)

    def fit_transform(self, x: np.ndarray) -> np.ndarray:
        self.fit(x)
        return self.transform(x)


## 7. Model 1: Logistic Regression

In [ ]:
class LogisticRegressionModel:
    def __init__(
        self,
        learning_rate: float = 0.05,
        n_iterations: int = 5000,
        l2: float = 0.0,
    ):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.l2 = l2
        self.weights: np.ndarray | None = None
        self.bias = 0.0

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        z = np.clip(z, -500.0, 500.0)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, x: np.ndarray, y: np.ndarray) -> "LogisticRegressionModel":
        n_samples, n_features = x.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0
        y_float = y.astype(float)

        for _ in range(self.n_iterations):
            scores = np.dot(x, self.weights) + self.bias
            probabilities = self._sigmoid(scores)
            errors = probabilities - y_float
            grad_w = np.dot(x.T, errors) / n_samples
            if self.l2 > 0:
                grad_w += self.l2 * self.weights / n_samples
            grad_b = float(errors.mean())
            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b
        return self

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        if self.weights is None:
            raise RuntimeError("Model has not been fitted.")
        scores = np.dot(x, self.weights) + self.bias
        return self._sigmoid(scores)

    def predict(self, x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(x) >= threshold).astype(int)


## 8. Model 2: Decision Tree

In [ ]:
class DecisionTreeModel:
    def __init__(
        self,
        max_depth: int = 4,
        min_samples_split: int = 5,
        max_features: int | None = None,
        random_state: int | None = None,
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.tree: dict | None = None
        self.rng = np.random.RandomState(random_state)
        self.feature_importances_: np.ndarray | None = None

    def _gini(self, y: np.ndarray) -> float:
        if len(y) == 0:
            return 0.0
        p_one = float(np.mean(y))
        return 1.0 - p_one * p_one - (1.0 - p_one) * (1.0 - p_one)

    def _leaf(self, y: np.ndarray) -> dict:
        probability = float(np.mean(y)) if len(y) else 0.0
        prediction = 1 if probability >= 0.5 else 0
        return {"type": "leaf", "prediction": prediction, "probability": probability}

    def _candidate_features(self, n_features: int) -> np.ndarray:
        if self.max_features is None or self.max_features >= n_features:
            return np.arange(n_features)
        return self.rng.choice(n_features, size=self.max_features, replace=False)

    def _best_split(self, x: np.ndarray, y: np.ndarray) -> tuple[int, float, float] | None:
        n_samples, n_features = x.shape
        parent_gini = self._gini(y)
        best_gain = 0.0
        best_feature = -1
        best_threshold = 0.0

        for feature_idx in self._candidate_features(n_features):
            values = np.unique(x[:, feature_idx])
            if len(values) <= 1:
                continue
            thresholds = (values[:-1] + values[1:]) / 2.0
            for threshold in thresholds:
                left_mask = x[:, feature_idx] <= threshold
                right_mask = ~left_mask
                n_left = int(left_mask.sum())
                n_right = n_samples - n_left
                if n_left == 0 or n_right == 0:
                    continue

                left_gini = self._gini(y[left_mask])
                right_gini = self._gini(y[right_mask])
                weighted_gini = (n_left / n_samples) * left_gini + (
                    n_right / n_samples
                ) * right_gini
                gain = parent_gini - weighted_gini

                if gain > best_gain:
                    best_gain = gain
                    best_feature = int(feature_idx)
                    best_threshold = float(threshold)

        if best_feature == -1:
            return None
        return best_feature, best_threshold, best_gain

    def _build(self, x: np.ndarray, y: np.ndarray, depth: int) -> dict:
        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return self._leaf(y)

        split = self._best_split(x, y)
        if split is None:
            return self._leaf(y)

        feature_idx, threshold, gain = split
        left_mask = x[:, feature_idx] <= threshold
        right_mask = ~left_mask
        if self.feature_importances_ is not None:
            self.feature_importances_[feature_idx] += gain * len(y)

        return {
            "type": "node",
            "feature_idx": feature_idx,
            "threshold": threshold,
            "left": self._build(x[left_mask], y[left_mask], depth + 1),
            "right": self._build(x[right_mask], y[right_mask], depth + 1),
        }

    def fit(self, x: np.ndarray, y: np.ndarray) -> "DecisionTreeModel":
        self.feature_importances_ = np.zeros(x.shape[1], dtype=float)
        self.tree = self._build(x, y.astype(int), depth=0)
        total = float(self.feature_importances_.sum())
        if total > 0:
            self.feature_importances_ = self.feature_importances_ / total
        return self

    def _predict_one_proba(self, row: np.ndarray, node: dict) -> float:
        if node["type"] == "leaf":
            return float(node["probability"])
        if row[node["feature_idx"]] <= node["threshold"]:
            return self._predict_one_proba(row, node["left"])
        return self._predict_one_proba(row, node["right"])

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        if self.tree is None:
            raise RuntimeError("Tree has not been fitted.")
        return np.array([self._predict_one_proba(row, self.tree) for row in x])

    def predict(self, x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(x) >= threshold).astype(int)


## 9. Model 3: Random Forest

In [ ]:
class RandomForestModel:
    def __init__(
        self,
        n_estimators: int = 21,
        max_depth: int = 5,
        min_samples_split: int = 5,
        max_features: str | int = "sqrt",
        sample_ratio: float = 1.0,
        random_state: int = 42,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.sample_ratio = sample_ratio
        self.random_state = random_state
        self.trees: list[DecisionTreeModel] = []
        self.feature_importances_: np.ndarray | None = None

    def _resolve_max_features(self, n_features: int) -> int | None:
        if self.max_features == "sqrt":
            return max(1, int(math.sqrt(n_features)))
        if isinstance(self.max_features, int):
            return min(n_features, self.max_features)
        return None

    def fit(self, x: np.ndarray, y: np.ndarray) -> "RandomForestModel":
        rng = np.random.RandomState(self.random_state)
        n_samples, n_features = x.shape
        sample_size = max(1, int(self.sample_ratio * n_samples))
        max_features = self._resolve_max_features(n_features)
        self.trees = []

        for estimator_idx in range(self.n_estimators):
            sample_idx = rng.choice(n_samples, size=sample_size, replace=True)
            tree = DecisionTreeModel(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_features,
                random_state=self.random_state + estimator_idx,
            )
            tree.fit(x[sample_idx], y[sample_idx])
            self.trees.append(tree)

        importances = np.zeros(n_features, dtype=float)
        for tree in self.trees:
            if tree.feature_importances_ is not None:
                importances += tree.feature_importances_
        total = float(importances.sum())
        self.feature_importances_ = importances / total if total > 0 else importances
        return self

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        if not self.trees:
            raise RuntimeError("Forest has not been fitted.")
        all_probs = np.vstack([tree.predict_proba(x) for tree in self.trees])
        return all_probs.mean(axis=0)

    def predict(self, x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(x) >= threshold).astype(int)


## 10. Model 4: KNN

In [ ]:
class KNNModel:
    def __init__(self, k: int = 5, weighted: bool = False, eps: float = 1e-8):
        self.k = k
        self.weighted = weighted
        self.eps = eps
        self.x_train: np.ndarray | None = None
        self.y_train: np.ndarray | None = None

    def fit(self, x: np.ndarray, y: np.ndarray) -> "KNNModel":
        self.x_train = x.copy()
        self.y_train = y.astype(int).copy()
        return self

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        if self.x_train is None or self.y_train is None:
            raise RuntimeError("KNN has not been fitted.")
        probabilities = []
        for row in x:
            distances = np.sqrt(np.sum((self.x_train - row) ** 2, axis=1))
            neighbor_idx = np.argsort(distances)[: self.k]
            neighbor_labels = self.y_train[neighbor_idx]
            if self.weighted:
                weights = 1.0 / (distances[neighbor_idx] + self.eps)
                prob = float(np.sum(weights * neighbor_labels) / np.sum(weights))
            else:
                prob = float(np.mean(neighbor_labels))
            probabilities.append(prob)
        return np.array(probabilities)

    def predict(self, x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(x) >= threshold).astype(int)


## 11. Metrics and Evaluation Helpers

In [ ]:
def confusion_counts(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, int]:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    return {
        "tp": int(np.sum((y_true == 1) & (y_pred == 1))),
        "tn": int(np.sum((y_true == 0) & (y_pred == 0))),
        "fp": int(np.sum((y_true == 0) & (y_pred == 1))),
        "fn": int(np.sum((y_true == 1) & (y_pred == 0))),
    }


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    counts = confusion_counts(y_true, y_pred)
    tp = counts["tp"]
    tn = counts["tn"]
    fp = counts["fp"]
    fn = counts["fn"]
    total = len(y_true)
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    out = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    out.update(counts)
    return out


def roc_auc_manual(y_true: np.ndarray, scores: np.ndarray) -> float:
    y_true = y_true.astype(int)
    positives = np.sum(y_true == 1)
    negatives = np.sum(y_true == 0)
    if positives == 0 or negatives == 0:
        return float("nan")

    order = np.argsort(-scores)
    y_sorted = y_true[order]
    tps = np.cumsum(y_sorted == 1)
    fps = np.cumsum(y_sorted == 0)
    tpr = np.concatenate([[0.0], tps / positives, [1.0]])
    fpr = np.concatenate([[0.0], fps / negatives, [1.0]])
    return float(np.trapz(tpr, fpr))


## 12. Hyperparameter Tuning Helpers

In [ ]:
def stratified_train_validation_split(
    x: np.ndarray,
    y: np.ndarray,
    validation_size: float = 0.2,
    random_state: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.RandomState(random_state)
    train_indices: list[int] = []
    validation_indices: list[int] = []
    for label in np.unique(y):
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)
        n_val = max(1, int(round(len(label_indices) * validation_size)))
        validation_indices.extend(label_indices[:n_val].tolist())
        train_indices.extend(label_indices[n_val:].tolist())

    rng.shuffle(train_indices)
    rng.shuffle(validation_indices)
    return x[train_indices], x[validation_indices], y[train_indices], y[validation_indices]


def make_model(model_name: str, params: dict) -> object:
    if model_name == "Logistic Regression":
        return LogisticRegressionModel(**params)
    if model_name == "Decision Tree":
        return DecisionTreeModel(**params)
    if model_name == "Random Forest":
        return RandomForestModel(**params)
    if model_name == "KNN":
        return KNNModel(**params)
    raise ValueError(f"Unknown model: {model_name}")


def candidate_grid() -> dict[str, list[dict]]:
    return {
        "Logistic Regression": [
            {"learning_rate": 0.03, "n_iterations": 5000, "l2": 0.0},
            {"learning_rate": 0.05, "n_iterations": 6000, "l2": 0.01},
            {"learning_rate": 0.08, "n_iterations": 7000, "l2": 0.05},
        ],
        "Decision Tree": [
            {"max_depth": 3, "min_samples_split": 5, "random_state": 42},
            {"max_depth": 4, "min_samples_split": 5, "random_state": 42},
            {"max_depth": 5, "min_samples_split": 8, "random_state": 42},
        ],
        "Random Forest": [
            {
                "n_estimators": 15,
                "max_depth": 4,
                "min_samples_split": 5,
                "max_features": "sqrt",
                "sample_ratio": 1.0,
                "random_state": 42,
            },
            {
                "n_estimators": 25,
                "max_depth": 5,
                "min_samples_split": 5,
                "max_features": "sqrt",
                "sample_ratio": 1.0,
                "random_state": 42,
            },
            {
                "n_estimators": 31,
                "max_depth": 5,
                "min_samples_split": 8,
                "max_features": "sqrt",
                "sample_ratio": 1.0,
                "random_state": 42,
            },
        ],
        "KNN": [
            {"k": 3, "weighted": False},
            {"k": 5, "weighted": False},
            {"k": 7, "weighted": False},
            {"k": 9, "weighted": True},
        ],
    }


def tune_models(
    x_train: np.ndarray,
    y_train: np.ndarray,
    feature_names: list[str],
) -> pd.DataFrame:
    x_internal_train, x_internal_val, y_internal_train, y_internal_val = (
        stratified_train_validation_split(x_train, y_train, validation_size=0.2)
    )
    scaler = StandardScalerModel()
    x_internal_train_scaled = scaler.fit_transform(x_internal_train)
    x_internal_val_scaled = scaler.transform(x_internal_val)

    rows: list[dict] = []
    for model_name, configs in candidate_grid().items():
        thresholds = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6] if model_name == "Logistic Regression" else [0.5]
        for params in configs:
            model = make_model(model_name, params)
            model.fit(x_internal_train_scaled, y_internal_train)
            scores = model.predict_proba(x_internal_val_scaled)
            for threshold in thresholds:
                preds = (scores >= threshold).astype(int)
                metrics = binary_metrics(y_internal_val, preds)
                rows.append(
                    {
                        "model": model_name,
                        "params": json.dumps(params, sort_keys=True),
                        "threshold": threshold,
                        "validation_accuracy": metrics["accuracy"],
                        "validation_precision": metrics["precision"],
                        "validation_recall": metrics["recall"],
                        "validation_f1": metrics["f1"],
                        "feature_count": len(feature_names),
                    }
                )

    tuning_df = pd.DataFrame(rows)
    tuning_df = tuning_df.sort_values(
        by=["model", "validation_f1", "validation_recall", "validation_accuracy"],
        ascending=[True, False, False, False],
    )
    return tuning_df


def best_config_per_model(tuning_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_name in sorted(tuning_df["model"].unique()):
        model_rows = tuning_df[tuning_df["model"] == model_name].copy()
        model_rows = model_rows.sort_values(
            by=["validation_f1", "validation_recall", "validation_accuracy"],
            ascending=[False, False, False],
        )
        rows.append(model_rows.iloc[0].to_dict())
    return pd.DataFrame(rows)


def evaluate_final_models(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_dev: np.ndarray,
    y_dev: np.ndarray,
    best_configs: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, object], StandardScalerModel]:
    scaler = StandardScalerModel()
    x_train_scaled = scaler.fit_transform(x_train)
    x_dev_scaled = scaler.transform(x_dev)

    rows = []
    trained_models: dict[str, object] = {}
    for _, row in best_configs.iterrows():
        model_name = str(row["model"])
        params = json.loads(str(row["params"]))
        threshold = float(row["threshold"])
        model = make_model(model_name, params)
        model.fit(x_train_scaled, y_train)
        scores = model.predict_proba(x_dev_scaled)
        preds = (scores >= threshold).astype(int)
        metrics = binary_metrics(y_dev, preds)
        rows.append(
            {
                "model": model_name,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "roc_auc": roc_auc_manual(y_dev, scores),
                "tp": metrics["tp"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "threshold": threshold,
                "params": json.dumps(params, sort_keys=True),
            }
        )
        trained_models[model_name] = model

    results_df = pd.DataFrame(rows)
    results_df = results_df.sort_values(
        by=["f1", "recall", "accuracy", "precision", "roc_auc"],
        ascending=False,
    )
    return results_df, trained_models, scaler


def permutation_importance(
    model: object,
    x_dev_scaled: np.ndarray,
    y_dev: np.ndarray,
    feature_names: list[str],
    threshold: float,
    random_state: int = 42,
    repeats: int = 10,
) -> pd.DataFrame:
    baseline_scores = model.predict_proba(x_dev_scaled)
    baseline_preds = (baseline_scores >= threshold).astype(int)
    baseline_f1 = binary_metrics(y_dev, baseline_preds)["f1"]
    rng = np.random.RandomState(random_state)
    rows = []

    for idx, feature in enumerate(feature_names):
        drops = []
        for _ in range(repeats):
            x_permuted = x_dev_scaled.copy()
            x_permuted[:, idx] = rng.permutation(x_permuted[:, idx])
            perm_scores = model.predict_proba(x_permuted)
            perm_preds = (perm_scores >= threshold).astype(int)
            perm_f1 = binary_metrics(y_dev, perm_preds)["f1"]
            drops.append(baseline_f1 - perm_f1)
        rows.append(
            {
                "feature": feature,
                "mean_f1_drop": float(np.mean(drops)),
                "std_f1_drop": float(np.std(drops)),
            }
        )
    return pd.DataFrame(rows).sort_values(by="mean_f1_drop", ascending=False)


## 13. Full Pipeline and Figure Generation

In [ ]:
def create_figures(
    train_df: pd.DataFrame,
    final_results: pd.DataFrame,
    y_dev: np.ndarray,
    dev_preds: np.ndarray,
    perm_importance: pd.DataFrame,
    analysis_df: pd.DataFrame,
    figures_dir: Path,
) -> None:
    sns.set_theme(style="whitegrid")

    plt.figure(figsize=(6, 4))
    ax = sns.countplot(data=train_df, x="label", hue="label", palette=["#4C78A8", "#F58518"], legend=False)
    ax.set_title("Target Distribution in Final Training Set")
    ax.set_xlabel("Scholarship label")
    ax.set_ylabel("Number of students")
    plt.tight_layout()
    plt.savefig(figures_dir / "target_distribution.png", dpi=180)
    plt.close()

    corr_cols = FINAL_BASE_FEATURES + ["label"]
    plt.figure(figsize=(8, 6))
    sns.heatmap(train_df[corr_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(figures_dir / "correlation_heatmap.png", dpi=180)
    plt.close()

    plt.figure(figsize=(7, 4))
    sns.boxplot(data=train_df, x="label", y="gpa", hue="label", palette=["#4C78A8", "#F58518"], legend=False)
    plt.title("GPA by Scholarship Label")
    plt.xlabel("Scholarship label")
    plt.ylabel("GPA")
    plt.tight_layout()
    plt.savefig(figures_dir / "gpa_by_label.png", dpi=180)
    plt.close()

    plt.figure(figsize=(7, 4))
    sns.boxplot(
        data=train_df,
        x="label",
        y="part_time_job_hours",
        hue="label",
        palette=["#4C78A8", "#F58518"],
        legend=False,
    )
    plt.title("Part-Time Job Hours by Scholarship Label")
    plt.xlabel("Scholarship label")
    plt.ylabel("Part-time job hours")
    plt.tight_layout()
    plt.savefig(figures_dir / "part_time_by_label.png", dpi=180)
    plt.close()

    plot_df = final_results.sort_values(by="f1", ascending=True)
    plt.figure(figsize=(7, 4))
    plt.barh(plot_df["model"], plot_df["f1"], color="#54A24B")
    plt.xlim(0, 1.02)
    plt.title("Final Model Comparison by F1-score")
    plt.xlabel("F1-score")
    plt.tight_layout()
    plt.savefig(figures_dir / "model_comparison_f1.png", dpi=180)
    plt.close()

    counts = confusion_counts(y_dev, dev_preds)
    cm = np.array([[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]])
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Reds",
        xticklabels=["Predicted 0", "Predicted 1"],
        yticklabels=["Actual 0", "Actual 1"],
    )
    plt.title("Confusion Matrix of Selected Model")
    plt.tight_layout()
    plt.savefig(figures_dir / "selected_confusion_matrix.png", dpi=180)
    plt.close()

    top_importance = perm_importance.head(10).sort_values(by="mean_f1_drop", ascending=True)
    plt.figure(figsize=(7, 5))
    plt.barh(top_importance["feature"], top_importance["mean_f1_drop"], color="#B279A2")
    plt.title("Permutation Importance of Selected Model")
    plt.xlabel("Mean F1 drop after permutation")
    plt.tight_layout()
    plt.savefig(figures_dir / "permutation_importance.png", dpi=180)
    plt.close()

    error_df = analysis_df[~analysis_df["is_correct"]]
    correct_df = analysis_df[analysis_df["is_correct"]]
    if len(error_df) > 0:
        means = pd.DataFrame(
            {
                "misclassified": error_df[FINAL_BASE_FEATURES].mean(numeric_only=True),
                "correct": correct_df[FINAL_BASE_FEATURES].mean(numeric_only=True),
            }
        )
        means["difference"] = means["misclassified"] - means["correct"]
        means = means.sort_values(by="difference")
        plt.figure(figsize=(7, 4))
        plt.barh(means.index, means["difference"], color="#E45756")
        plt.axvline(0, color="black", linewidth=0.8)
        plt.title("Mean Feature Difference: Errors vs Correct Predictions")
        plt.xlabel("Misclassified mean minus correct mean")
        plt.tight_layout()
        plt.savefig(figures_dir / "error_feature_difference.png", dpi=180)
        plt.close()


def run_training_pipeline(final_dir: Path, hidden_path: Path | str | None = None, save_outputs: bool = True) -> dict:
    final_dir = Path(final_dir).resolve()
    train_path = final_dir / "ml_dataset_v2_train.csv"
    dev_path = final_dir / "ml_dataset_v2_dev.csv"
    if hidden_path is None:
        hidden_path = final_dir / "hidden_test_data.csv"
    else:
        hidden_path = Path(hidden_path)
        if not hidden_path.is_absolute():
            hidden_path = final_dir / hidden_path

    for path, label in [
        (train_path, "training CSV"),
        (dev_path, "development CSV"),
        (hidden_path, "hidden/test CSV"),
    ]:
        if not path.exists():
            raise FileNotFoundError(
                f"Missing {label}: {path}. Put the CSV files in the same folder as the notebook."
            )

    figures_dir = final_dir / "figures"
    results_dir = final_dir / "results"
    if save_outputs:
        figures_dir.mkdir(parents=True, exist_ok=True)
        results_dir.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_csv(train_path)
    dev_df = pd.read_csv(dev_path)
    hidden_df = pd.read_csv(hidden_path)

    validate_columns(train_df, ["id"] + FINAL_BASE_FEATURES + ["label"], "final train")
    validate_columns(dev_df, ["id"] + FINAL_BASE_FEATURES + ["label"], "final dev")
    validate_columns(hidden_df, ["id"] + MIDTERM_BASE_FEATURES, "hidden test")

    medians = feature_medians_from_train(train_df)

    x_train_baseline = build_feature_frame(
        train_df, medians, include_engineered=False, feature_cols=MIDTERM_BASE_FEATURES
    ).to_numpy(dtype=float)
    x_dev_baseline = build_feature_frame(
        dev_df, medians, include_engineered=False, feature_cols=MIDTERM_BASE_FEATURES
    ).to_numpy(dtype=float)
    y_train = train_df["label"].to_numpy(dtype=int)
    y_dev = dev_df["label"].to_numpy(dtype=int)

    baseline_scaler = StandardScalerModel()
    x_train_baseline_scaled = baseline_scaler.fit_transform(x_train_baseline)
    x_dev_baseline_scaled = baseline_scaler.transform(x_dev_baseline)
    baseline_model = LogisticRegressionModel(learning_rate=0.05, n_iterations=6000, l2=0.01)
    baseline_model.fit(x_train_baseline_scaled, y_train)
    baseline_scores = baseline_model.predict_proba(x_dev_baseline_scaled)
    baseline_preds = (baseline_scores >= 0.5).astype(int)
    baseline_metrics = binary_metrics(y_dev, baseline_preds)
    baseline_result = {
        "model": "Baseline Logistic Regression",
        "features": "Midterm feature set only",
        "accuracy": baseline_metrics["accuracy"],
        "precision": baseline_metrics["precision"],
        "recall": baseline_metrics["recall"],
        "f1": baseline_metrics["f1"],
        "roc_auc": roc_auc_manual(y_dev, baseline_scores),
        "tp": baseline_metrics["tp"],
        "tn": baseline_metrics["tn"],
        "fp": baseline_metrics["fp"],
        "fn": baseline_metrics["fn"],
    }

    x_train = build_feature_frame(train_df, medians, include_engineered=True).to_numpy(dtype=float)
    x_dev = build_feature_frame(dev_df, medians, include_engineered=True).to_numpy(dtype=float)

    tuning_df = tune_models(x_train, y_train, IMPROVED_FEATURES)
    best_configs = best_config_per_model(tuning_df)
    final_results, trained_models, final_scaler = evaluate_final_models(
        x_train, y_train, x_dev, y_dev, best_configs
    )

    selected_row = final_results.iloc[0].to_dict()
    selected_model_name = str(selected_row["model"])
    selected_threshold = float(selected_row["threshold"])
    selected_model = trained_models[selected_model_name]
    x_train_scaled = final_scaler.transform(x_train)
    x_dev_scaled = final_scaler.transform(x_dev)
    dev_scores = selected_model.predict_proba(x_dev_scaled)
    dev_preds = (dev_scores >= selected_threshold).astype(int)

    analysis_df = dev_df.copy()
    analysis_df["label_pred"] = dev_preds
    analysis_df["score"] = dev_scores
    analysis_df["is_correct"] = analysis_df["label"] == analysis_df["label_pred"]
    error_cases = analysis_df[~analysis_df["is_correct"]].copy()

    perm_importance = permutation_importance(
        selected_model, x_dev_scaled, y_dev, IMPROVED_FEATURES, selected_threshold
    )

    logistic_model = trained_models.get("Logistic Regression")
    logistic_coefficients = pd.DataFrame()
    if isinstance(logistic_model, LogisticRegressionModel) and logistic_model.weights is not None:
        logistic_coefficients = pd.DataFrame(
            {
                "feature": IMPROVED_FEATURES,
                "coefficient": logistic_model.weights,
                "absolute_coefficient": np.abs(logistic_model.weights),
            }
        ).sort_values(by="absolute_coefficient", ascending=False)

    rf_model = trained_models.get("Random Forest")
    rf_importances = pd.DataFrame()
    if isinstance(rf_model, RandomForestModel) and rf_model.feature_importances_ is not None:
        rf_importances = pd.DataFrame(
            {"feature": IMPROVED_FEATURES, "importance": rf_model.feature_importances_}
        ).sort_values(by="importance", ascending=False)

    hidden_aligned = build_feature_frame(hidden_df, medians, include_engineered=True)
    x_hidden_scaled = final_scaler.transform(hidden_aligned.to_numpy(dtype=float))
    hidden_scores = selected_model.predict_proba(x_hidden_scaled)
    hidden_preds = (hidden_scores >= selected_threshold).astype(int)
    submission = pd.DataFrame({"id": hidden_df["id"].astype(int), "label_pred": hidden_preds.astype(int)})

    hidden_test_metrics = None
    if "label" in hidden_df.columns:
        hidden_test_metrics = binary_metrics(hidden_df["label"].to_numpy(dtype=int), hidden_preds)
        hidden_test_metrics["roc_auc"] = roc_auc_manual(
            hidden_df["label"].to_numpy(dtype=int), hidden_scores
        )

    data_summary = {
        "train_shape": train_df.shape,
        "dev_shape": dev_df.shape,
        "hidden_shape": hidden_df.shape,
        "train_label_counts": train_df["label"].value_counts().sort_index().to_dict(),
        "dev_label_counts": dev_df["label"].value_counts().sort_index().to_dict(),
        "missing_train": int(train_df.isna().sum().sum()),
        "missing_dev": int(dev_df.isna().sum().sum()),
        "hidden_missing_final_features": [
            col for col in FINAL_BASE_FEATURES if col not in hidden_df.columns
        ],
        "feature_medians": medians,
    }

    if save_outputs:
        pd.DataFrame([baseline_result]).to_csv(results_dir / "baseline_result.csv", index=False)
        tuning_df.to_csv(results_dir / "tuning_results.csv", index=False)
        best_configs.to_csv(results_dir / "best_configs.csv", index=False)
        final_results.to_csv(results_dir / "model_comparison.csv", index=False)
        analysis_df.to_csv(results_dir / "dev_predictions_with_errors.csv", index=False)
        error_cases.to_csv(results_dir / "error_cases.csv", index=False)
        perm_importance.to_csv(results_dir / "permutation_importance.csv", index=False)
        logistic_coefficients.to_csv(results_dir / "logistic_coefficients.csv", index=False)
        rf_importances.to_csv(results_dir / "random_forest_importance.csv", index=False)
        submission.to_csv(final_dir / "predictions.csv", index=False)
        with (results_dir / "summary.json").open("w", encoding="utf-8") as f:
            json.dump(
                {
                    "data_summary": data_summary,
                    "selected_model": selected_row,
                    "hidden_test_metrics": hidden_test_metrics,
                },
                f,
                indent=2,
                default=str,
            )
        create_figures(
            train_df,
            final_results,
            y_dev,
            dev_preds,
            perm_importance,
            analysis_df,
            figures_dir,
        )

    return {
        "train_df": train_df,
        "dev_df": dev_df,
        "hidden_df": hidden_df,
        "baseline_result": baseline_result,
        "tuning_df": tuning_df,
        "best_configs": best_configs,
        "final_results": final_results,
        "selected_row": selected_row,
        "selected_model_name": selected_model_name,
        "selected_threshold": selected_threshold,
        "analysis_df": analysis_df,
        "error_cases": error_cases,
        "permutation_importance": perm_importance,
        "logistic_coefficients": logistic_coefficients,
        "rf_importances": rf_importances,
        "submission": submission,
        "hidden_test_metrics": hidden_test_metrics,
        "data_summary": data_summary,
        "figures_dir": figures_dir,
        "results_dir": results_dir,
    }


## 14. Run Full Training Pipeline

In [ ]:
artifacts = run_training_pipeline(FINAL_DIR, HIDDEN_PATH, save_outputs=True)
print('Selected model:', artifacts['selected_model_name'])
display(artifacts['final_results'])
display(pd.DataFrame([artifacts['baseline_result']]))

## 15. Error Analysis

In [ ]:
error_cases = artifacts['error_cases']
print('Number of dev errors:', len(error_cases))
display(error_cases[['id','label','label_pred','score','gpa','exam_score','part_time_job_hours']].head(20))
if len(error_cases) > 0:
    display(error_cases[FINAL_BASE_FEATURES].describe().T)

## 16. Model Interpretation

In [ ]:
display(artifacts['permutation_importance'].head(10))
display(artifacts['logistic_coefficients'].head(10))
display(artifacts['rf_importances'].head(10))

## 17. Generate Submission Predictions

In [ ]:
submission = artifacts['submission']
display(submission.head())
print('Submission shape:', submission.shape)
print('Label distribution:', submission['label_pred'].value_counts().sort_index().to_dict())
submission.to_csv(FINAL_DIR / 'predictions.csv', index=False)
print('Saved:', FINAL_DIR / 'predictions.csv')